## Tutorial 3 - Preparing for production

To deal with any steric clashes that could have been present in the input system or introduced in the coarse-graining stage or in bilyaer building, we need to energy minimise the system. For coarse-grained systems, this will normally suffice to be able to seed a production run, but care should be taken to disregard the start of the production simulation as the system equilibrates.

The most well used simulation engine for Martini simulations is gromacs, which we will use here. There is now an implimentation of [Martini in OpenMM](https://www.cell.com/biophysj/fulltext/S0006-3495(23)00237-0?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS0006349523002370%3Fshowall%3Dtrue) which can also be used.   

The `grompp` step in gromacs is used to prepare a production file, which is an output in the form of a `.tpr` file. The inputs are:
- `-c` a coordinate file (`.pdb` or`.gro`)
- `-p` the topology file, showing what is contained within the system and information about the forcefield/parameter files
- `-f` an `.mdp` file, which contains the setting for this simulation run
- `-maxwarn` which suppresses warnings. **This should not be used unless you know what you are doing**, but used below to dismiss warnings about atom name changes (which happened with new lipid parameters) and another warning that we can disregard for now

We can have a look at what is contained within this energy minimisation file:

In [1]:
!head mdps/em.mdp

integrator = steep
nsteps     = 5000
emtol      = 1000
emstep     = 0.001


- `integrator` defines what type of simulation is performed, in this case it uses the steepest decents algorithm to perform energy minimisation  
- `nsteps` defines the maximum number of steps performed  
- `emtol` is the cutoff where we define the system as energy minimised, a target energy (in kJ mol<sup>-1</sup>)  
- `emstep` is the largest displacement allowed per step performed (in nm)  

We can now use this file and our files we created in the previous step to generate a `.tpr` file

In [2]:
%%bash 

cp ../tutorial_2/system.gro ../tutorial_2/topol.top .

cp ../tutorial_2/*.itp  itps/

gmx grompp -f mdps/em.mdp -c system.gro -p topol.top -o em.tpr -maxwarn 2

                :-) GROMACS - gmx grompp, 2025.4-conda_forge (-:

Executable:   /storage/chem/lfsmgr/SRG/mambaforge/envs/gmx2025.4/bin.AVX2_256/gmx
Data prefix:  /storage/chem/lfsmgr/SRG/mambaforge/envs/gmx2025.4
Working dir:  /storage/lfsmgr_grp/nttpkm/CCPBioSim_training/tutorial_3
Command line:
  gmx grompp -f mdps/em.mdp -c system.gro -p topol.top -o em.tpr -maxwarn 2

Number of degrees of freedom in T-Coupling group rest is 60697.00
The integrator does not provide a ensemble temperature, there is no system ensemble temperature

NOTE 1 [file mdps/em.mdp]:
  You are using a plain Coulomb cut-off, which might produce artifacts.
  You might want to consider using PME electrostatics.



There was 1 NOTE

GROMACS reminds you: "Do You Have Sex Maniacs or Schizophrenics or Astrophysicists in Your Family?" (Gogol Bordello)



Setting the LD random seed to -23726361

Generated 844 of the 356590 non-bonded parameter combinations

Excluding 1 bonded neighbours molecule type 'VDAC1_0'

Excluding 1 bonded neighbours molecule type 'POPC'

Excluding 1 bonded neighbours molecule type 'POPE'

Excluding 1 bonded neighbours molecule type 'POPI'

Excluding 1 bonded neighbours molecule type 'POPC'

Excluding 1 bonded neighbours molecule type 'POPE'

Excluding 1 bonded neighbours molecule type 'POPI'

Excluding 1 bonded neighbours molecule type 'W'

Excluding 1 bonded neighbours molecule type 'NA'

Excluding 1 bonded neighbours molecule type 'CL'

Cleaning up constraints and constant bonded interactions with virtual sites

Cleaning up constraints and constant bonded interactions with virtual sites
Analysing residue names:
There are:   283    Protein residues
There are: 11256      Other residues
Analysing Protein...
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...

This run will gene

We can now run our energy minimisation. We do this with the `gmx mdrun` command, which takes the following inputs:

- `-deffnm` sets the default file names and uses this to find the .tpr file
- `-v` to be verbose and print all the steps it is taking (and when it might finish)
- `-ntmpi` number of thread-MPI ranks to start

In [3]:
%%bash

gmx mdrun -deffnm em -v -ntmpi 1

                :-) GROMACS - gmx mdrun, 2025.4-conda_forge (-:

Executable:   /storage/chem/lfsmgr/SRG/mambaforge/envs/gmx2025.4/bin.AVX2_256/gmx
Data prefix:  /storage/chem/lfsmgr/SRG/mambaforge/envs/gmx2025.4
Working dir:  /storage/lfsmgr_grp/nttpkm/CCPBioSim_training/tutorial_3
Command line:
  gmx mdrun -deffnm em -v -ntmpi 1

The current CPU can measure timings more accurately than the code in
gmx mdrun was configured to use. This might affect your simulation
speed as accurate timings are needed for load-balancing.
Please consider rebuilding gmx mdrun with the GMX_USE_RDTSCP=ON CMake option.
Reading file em.tpr, VERSION 2025.4-conda_forge (single precision)

Update groups can not be used for this system because an incompatible virtual site type is used

1 GPU selected for this run.
Mapping of GPU IDs to the 1 GPU task in the 1 rank on this node:
  PP:0
PP tasks will do (non-perturbed) short-ranged interactions on the GPU
PP task will update and constrain coordinates on the CPU
Usi

We now have an energy minimised system! We can have a look at this in VMD and hopefully you can see the difference:

Do this **out of the notebook** in your own terminal:

```$vmd em.gro```

We can now use this to generate our production simulation. The `.mdp` file listing the settings is a lot longer and more complicated, we can see this below. 

In [4]:
%%bash

head -n 44 mdps/5us-martini.mdp

integrator           = md			
tinit                = 0.0			
dt                   = 0.02
nsteps               = 250000000
nstxout              = 0
nstvout              = 0
nstfout              = 0
nstlog               = 50000
nstenergy            = 50000
nstxout-compressed   = 50000
compressed-x-precision = 10000
nstlist              = 10
ns_type              = grid
pbc                  = xyz
rlist                = 1.35
verlet-buffer-tolerance  = -1
coulombtype          = Reaction_field
rcoulomb_switch      = 0.0
rcoulomb             = 1.1
epsilon_r            = 15 
vdw_type             = cutoff 
rvdw_switch          = 0.9
rvdw                 = 1.1
cutoff-scheme        = verlet
coulomb-modifier     = Potential-shift
vdw-modifier         = Potential-shift
epsilon_rf           = 0  
tcoupl               = v-rescale 
tc-grps              = Protein LIPID SOL_ION
tau_t                = 1.0 1.0 1.0
ref_t                = 310 310 310
Pcoupl               = c-rescale
Pcoupltype           = semi

We can highlight the most important settings here

- `integrator` this time is md - which will use an algorithm for integrating Newton's equation of motion (in this case using a leap-frog algorithm).
- `dt` which is the time step used by the integrator. For atomistic simulations this is often 2 fs, but with Martini this can be extended to 20 fs.
- `nsteps` which, with the md integrator, is the number of steps that will happen. As we are now using a time based integrator, we can calculate that 250000000 x 2 fs = 5 $\mu$s.
- `nstxout-compressed` is how many steps between saving coordinates into the `.xtc` format. As it stands, it is saved every nanosecond.
- `Pcoupltype` is the type of isotropy used for the pressure coupling. For soluable simulations, this will usually be set to isotropic where each dimension (x,y and z) would be treated uniquely. When there is a membrane present, the x and y dimensions are intrisically coupled, so we need to use the semiisotropic pressure coupling setting.
- `Pcouple` is the pressure coupling type to use, in this case the C-rescale algorithm.
- `tcoupl` is the tempreature coupling type, in this case the v-rescale algorithm.
- `tc-groups` specifies groups to seperate tempreature baths. Seperating into similarly mobile parts can help excessive energies accumulating in one component.

To be able to determine the groups used in `tc-groups` we need to make an index file. We can do that using a gromacs command:

In [5]:
%%bash

gmx make_ndx -f em.gro -o sys.ndx << EOF
rPOPC|rPOPE|rPOPI
name 18 LIPID
rW|rION
name 19 SOL_ION

q
EOF

               :-) GROMACS - gmx make_ndx, 2025.4-conda_forge (-:

Executable:   /storage/chem/lfsmgr/SRG/mambaforge/envs/gmx2025.4/bin.AVX2_256/gmx
Data prefix:  /storage/chem/lfsmgr/SRG/mambaforge/envs/gmx2025.4
Working dir:  /storage/lfsmgr_grp/nttpkm/CCPBioSim_training/tutorial_3
Command line:
  gmx make_ndx -f em.gro -o sys.ndx


Reading structure file

GROMACS reminds you: "I wanted to make a clever chemistry joke, but the best ones Argon." (39.948)



Going to read 0 old index file(s)
Analysing residue names:
There are:   283    Protein residues
There are: 11256      Other residues
Analysing Protein...
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...

  0 System              : 20482 atoms
  1 Protein             :   649 atoms
  2 Protein-H           :   649 atoms
  3 C-alpha             :     0 atoms
  4 Backbone            :     0 atoms
  5 MainChain           :     0 atoms
  6 MainChain+Cb        :     0 atoms
  7 MainChain+H         :     0 atoms
  8 SideChain           :   649 atoms
  9 SideChain-H         :   649 atoms
 10 Prot-Masses         :   649 atoms
 11 non-Protein         : 19833 atoms
 12 Other               : 19833 atoms
 13 POPC                :  5052 atoms
 14 POPE                :  2976 atoms
 15 POPI                :  1305 atoms
 16 W                   : 10190 atoms
 17 ION                 :   310 atoms

 nr : group      '!': not  'name' nr name   'splitch' nr    Enter: list 

Since the Protein is already listed in the index file, we do not need to specify this further. We need to group the lipids (**r**esidue POPE |-or etc) together and give this the appropriate name, same with the lipids and ions.

We are now ready to simulate our system! Depending on what you are studying, the number of repeats and length of simulation could differ. For investigating something such as lipid-protein interaction I would usually start with 5 x 5 $\mu$s simulations, especially for a system of this size. We can set up one simulation below:

In [6]:
%%bash

gmx grompp -f mdps/5us-martini.mdp -c em.gro -p topol.top -n sys.ndx -o md.tpr

                :-) GROMACS - gmx grompp, 2025.4-conda_forge (-:

Executable:   /storage/chem/lfsmgr/SRG/mambaforge/envs/gmx2025.4/bin.AVX2_256/gmx
Data prefix:  /storage/chem/lfsmgr/SRG/mambaforge/envs/gmx2025.4
Working dir:  /storage/lfsmgr_grp/nttpkm/CCPBioSim_training/tutorial_3
Command line:
  gmx grompp -f mdps/5us-martini.mdp -c em.gro -p topol.top -n sys.ndx -o md.tpr

Ignoring obsolete mdp entry 'ns_type'

NOTE 1 [file mdps/5us-martini.mdp]:
  verlet-buffer-pressure-tolerance is ignored when verlet-buffer-tolerance
  < 0

Number of degrees of freedom in T-Coupling group Protein is 1722.91
Number of degrees of freedom in T-Coupling group LIPID is 27475.64
Number of degrees of freedom in T-Coupling group SOL_ION is 31498.44

There was 1 NOTE

GROMACS reminds you: "Somewhere, something incredible is waiting to be known." (Carl Sagan)



Setting the LD random seed to 1839853566

Generated 844 of the 356590 non-bonded parameter combinations

Excluding 1 bonded neighbours molecule type 'VDAC1_0'

Excluding 1 bonded neighbours molecule type 'POPC'

Excluding 1 bonded neighbours molecule type 'POPE'

Excluding 1 bonded neighbours molecule type 'POPI'

Excluding 1 bonded neighbours molecule type 'POPC'

Excluding 1 bonded neighbours molecule type 'POPE'

Excluding 1 bonded neighbours molecule type 'POPI'

Excluding 1 bonded neighbours molecule type 'W'

Excluding 1 bonded neighbours molecule type 'NA'

Excluding 1 bonded neighbours molecule type 'CL'

Setting gen_seed to -41978113

Velocities were taken from a Maxwell distribution at 310 K

Cleaning up constraints and constant bonded interactions with virtual sites

Cleaning up constraints and constant bonded interactions with virtual sites

This run will generate roughly 510 Mb of data


Because of time restraints, here is a simulation I prepared earlier so we can move on to analysis of coarse-grained simulations